# TRABAJO PRACTICO INTEGRAL N°1

## Integrantes:

**CRISTIAN DAMIAN FORTUNESKY BARRIOS**

**MATIAS DE VIVO**

## Actividad 1: Imagen obtenida mediante camara oscura


In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 5)


In [ ]:
# --- Configuración de rutas para Actividades 1, 2 y 3 ---

# Definimos la carpeta base del TFI para que todas las rutas sean reproducibles.
carpeta_tfi = Path("005 - TFI_1")

# Definimos la carpeta de entrada de imágenes.
carpeta_imagenes_tfi = carpeta_tfi / "img_input_tfi1"

# Creamos la carpeta si no existe para evitar errores de lectura/escritura.
carpeta_imagenes_tfi.mkdir(parents=True, exist_ok=True)

# Definimos carpeta de salidas para resultados procesados.
carpeta_salidas_tfi = carpeta_tfi / "img_output_tfi1"
carpeta_salidas_tfi.mkdir(parents=True, exist_ok=True)

print(f"Carpeta de imágenes: {carpeta_imagenes_tfi.resolve()}")
print(f"Carpeta de salidas: {carpeta_salidas_tfi.resolve()}")

## Actividad 1: Imagen obtenida mediante cámara oscura

**Origen de la imagen:** `img_input_tfi1/img-camara-oscura.jpg`  
**Salida final solicitada:** `img_output_tfi1/img-camara-oscura-final.jpg`

Esta actividad no parte de una lista de filtros, sino de un diagnóstico inicial. La imagen de cámara oscura suele presentar subexposición, bajo contraste, iluminación no uniforme, ruido por poca luz y posible desenfoque/movimiento. Por eso se comparan dos estrategias justificadas y se conserva como salida la versión que mejora la legibilidad sin destruir información importante.

> Nota de reproducibilidad: para cumplir la consigna del profesor, la fotografía de entrada debe ser propia, tomada con cámara oscura y de un objeto real. No se debe reemplazar por una imagen descargada de internet.


## Actividad 1 — Diagnóstico antes de filtrar

La restauración de la imagen de cámara oscura se plantea como un **pipeline justificado por diagnóstico**, no como una lista de filtros aplicados al azar.

**Imagen de entrada esperada:** `005 - TFI_1/img_input_tfi1/img-camara-oscura.jpg`  
**Imagen final generada:** `005 - TFI_1/img_output_tfi1/img-camara-oscura-final.jpg`

### Preguntas de diagnóstico inicial

| Pregunta | Respuesta técnica para esta imagen |
|---|---|
| ¿La imagen está oscura o clara? | Está **subexpuesta**: predominan niveles bajos/intermedios y no aparecen blancos reales. |
| ¿Tiene bajo contraste? | Sí. El objeto central se diferencia poco del fondo y el rango dinámico útil es reducido. |
| ¿Hay ruido? | Sí. Se observa granulado en zonas oscuras y posible ruido digital por amplificación en baja luz. |
| ¿La iluminación es desigual? | Sí. La zona útil queda concentrada en el centro y el fondo oscuro domina gran parte del cuadro. |
| ¿Existe deformación geométrica? | No se corrige una deformación geométrica fuerte en este pipeline. La degradación dominante es óptica/tonal, no de perspectiva. |
| ¿Hay manchas o daños localizados? | No se detecta una mancha puntual que justifique inpainting; el problema es global/local de captura. |
| ¿Qué parte importa? | La **región de interés central**, donde aparece la proyección de la cámara oscura. |

### Diagnóstico visual propio

El problema principal no es únicamente el ruido: la degradación dominante es la combinación de **desenfoque óptico + subexposición + bajo contraste**. El desenfoque no puede recuperarse por completo porque la información fina no fue registrada con nitidez en la captura. Por eso el objetivo realista es **mejorar legibilidad e interpretabilidad**, evitando inventar bordes falsos.

**Decisiones descartadas desde el diagnóstico:**

- No aplicar *sharpening* agresivo, porque amplificaría ruido, halos y bordes artificiales.
- No aplicar ecualización global fuerte como solución final, porque puede levantar demasiado el fondo y exagerar ruido/banding.


## Carga de imagen y funciones auxiliares

La celda siguiente deja preparado el flujo reproducible. Si la imagen todavía no está cargada en la carpeta de entrada, el cuaderno no falla: muestra una advertencia y mantiene documentado el pipeline para ejecutarlo cuando se incorpore la fotografía propia solicitada por la consigna.

In [ ]:
# --- Actividad 1: rutas y utilidades ---

nombre_imagen_camara = "img-camara-oscura.jpg"
ruta_camara = carpeta_imagenes_tfi / nombre_imagen_camara
ruta_final_camara = carpeta_salidas_tfi / "img-camara-oscura-final.jpg"
ruta_comparacion_camara = carpeta_salidas_tfi / "img-camara-oscura-comparacion.jpg"
ruta_histogramas_camara = carpeta_salidas_tfi / "img-camara-oscura-histogramas.jpg"


def cargar_rgb(ruta):
    """Carga una imagen con OpenCV y la devuelve en RGB para visualizar con Matplotlib."""
    imagen_bgr = cv2.imread(str(ruta), cv2.IMREAD_COLOR)
    if imagen_bgr is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {ruta}")
    return cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)


def guardar_rgb(ruta, imagen_rgb, calidad=95):
    """Guarda una imagen RGB en disco usando codificación BGR de OpenCV."""
    ruta.parent.mkdir(parents=True, exist_ok=True)
    imagen_bgr = cv2.cvtColor(imagen_rgb, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(ruta), imagen_bgr, [cv2.IMWRITE_JPEG_QUALITY, calidad])


def mostrar_imagen(imagen, titulo="Imagen", cmap=None):
    plt.figure(figsize=(8, 5))
    plt.imshow(imagen, cmap=cmap)
    plt.title(titulo)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


def mostrar_histograma_gris(imagen_gris, titulo="Histograma"):
    plt.figure(figsize=(8, 4))
    plt.hist(imagen_gris.ravel(), bins=256, range=(0, 255), color="dimgray")
    plt.title(titulo)
    plt.xlabel("Nivel de intensidad")
    plt.ylabel("Cantidad de píxeles")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


def calcular_metricas_gris(imagen_gris):
    """Calcula métricas simples para sostener el diagnóstico y la comparación."""
    hist = cv2.calcHist([imagen_gris], [0], None, [256], [0, 256]).ravel()
    prob = hist / max(hist.sum(), 1)
    prob = prob[prob > 0]

    p1, p5, p95, p99 = np.percentile(imagen_gris, [1, 5, 95, 99])
    mediana_filtrada = cv2.medianBlur(imagen_gris, 3)
    ruido_estimado = float(np.std(imagen_gris.astype(np.float32) - mediana_filtrada.astype(np.float32)))

    return {
        "brillo_medio": float(np.mean(imagen_gris)),
        "contraste_std": float(np.std(imagen_gris)),
        "p1": float(p1),
        "p5": float(p5),
        "p95": float(p95),
        "p99": float(p99),
        "rango_dinamico_p1_p99": float(p99 - p1),
        "nitidez_laplaciano": float(cv2.Laplacian(imagen_gris, cv2.CV_64F).var()),
        "entropia": float(-np.sum(prob * np.log2(prob))),
        "ruido_estimado": ruido_estimado,
        "banding_vertical_estimado": float(np.std(np.mean(imagen_gris, axis=0))),
    }


def imprimir_metricas(nombre, metricas):
    print(f"\n{nombre}")
    print("-" * len(nombre))
    for clave, valor in metricas.items():
        print(f"{clave:28s}: {valor:8.3f}")


if ruta_camara.exists():
    imagen_camara_rgb = cargar_rgb(ruta_camara)
    imagen_camara_gris = cv2.cvtColor(imagen_camara_rgb, cv2.COLOR_RGB2GRAY)
    print(f"Imagen cargada: {ruta_camara}")
    print(f"Dimensiones: {imagen_camara_rgb.shape[1]} x {imagen_camara_rgb.shape[0]} px")
    mostrar_imagen(imagen_camara_rgb, "Imagen original - cámara oscura")
    mostrar_histograma_gris(imagen_camara_gris, "Histograma original en escala de grises")
else:
    imagen_camara_rgb = None
    imagen_camara_gris = None
    print(f"ADVERTENCIA: no se encontró {ruta_camara}")
    print("Colocar allí la fotografía propia de cámara oscura para ejecutar el procesamiento completo.")

## Diagnóstico cuantitativo

El histograma permite traducir observaciones visuales a términos medibles:

- Si la media y los percentiles altos quedan bajos, hay **subexposición**.
- Si la desviación estándar y el rango entre percentiles son reducidos, hay **bajo contraste**.
- Si el residuo respecto de una mediana local es alto, hay **ruido fino**.
- Si el promedio por columnas varía de forma sistemática, puede existir **banding vertical** o iluminación estructurada.

Estas métricas no reemplazan el criterio visual, pero ayudan a justificar por qué se elige una mejora local y suave.

In [ ]:
if imagen_camara_gris is not None:
    metricas_original = calcular_metricas_gris(imagen_camara_gris)
    imprimir_metricas("Métricas de diagnóstico - original", metricas_original)

    diagnostico_textual = []
    if metricas_original["brillo_medio"] < 90:
        diagnostico_textual.append("subexposición / imagen oscura")
    if metricas_original["contraste_std"] < 45:
        diagnostico_textual.append("bajo contraste global")
    if metricas_original["ruido_estimado"] > 4:
        diagnostico_textual.append("ruido fino visible o probable")
    if metricas_original["nitidez_laplaciano"] < 80:
        diagnostico_textual.append("baja nitidez aparente / desenfoque")

    print("\nDiagnóstico automático orientativo:")
    print(", ".join(diagnostico_textual) if diagnostico_textual else "Sin degradación fuerte según umbrales simples.")
else:
    print("Diagnóstico cuantitativo pendiente: falta la imagen de entrada.")

## Recorte de región de interés (ROI)

Se recorta primero la región central útil porque el fondo oscuro ocupa mucho porcentaje de la imagen y domina las estadísticas. El recorte:

1. concentra el análisis en la proyección de la cámara oscura;
2. reduce la influencia del fondo negro en el histograma;
3. permite aplicar contraste local con menor riesgo de amplificar ruido irrelevante.

La función usa una detección automática basada en intensidades altas y componentes conectadas. Si esa detección no encuentra una región confiable, usa un recorte central conservador.

In [ ]:
def recortar_roi_camara(imagen_rgb, margen=35, percentil=88):
    """Recorta automáticamente la zona informativa; si no hay señal clara, usa recorte central."""
    gris = cv2.cvtColor(imagen_rgb, cv2.COLOR_RGB2GRAY)
    alto, ancho = gris.shape

    umbral = np.percentile(gris, percentil)
    mascara = (gris >= umbral).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, kernel, iterations=2)
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, kernel, iterations=1)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mascara, connectivity=8)
    if num_labels <= 1:
        usar_recorte_central = True
    else:
        areas = stats[1:, cv2.CC_STAT_AREA]
        idx = int(np.argmax(areas)) + 1
        area_relativa = stats[idx, cv2.CC_STAT_AREA] / float(alto * ancho)
        usar_recorte_central = area_relativa < 0.005

    if usar_recorte_central:
        x0, x1 = int(ancho * 0.15), int(ancho * 0.85)
        y0, y1 = int(alto * 0.15), int(alto * 0.85)
    else:
        x, y, w, h, _ = stats[idx]
        x0 = max(0, x - margen)
        y0 = max(0, y - margen)
        x1 = min(ancho, x + w + margen)
        y1 = min(alto, y + h + margen)

        # Evita recortes demasiado cerrados que podrían perder contexto de la proyección.
        min_w, min_h = int(ancho * 0.35), int(alto * 0.35)
        if (x1 - x0) < min_w:
            cx = (x0 + x1) // 2
            x0 = max(0, cx - min_w // 2)
            x1 = min(ancho, x0 + min_w)
        if (y1 - y0) < min_h:
            cy = (y0 + y1) // 2
            y0 = max(0, cy - min_h // 2)
            y1 = min(alto, y0 + min_h)

    roi = imagen_rgb[y0:y1, x0:x1]
    return roi, (x0, y0, x1, y1)


if imagen_camara_rgb is not None:
    imagen_roi_rgb, caja_roi = recortar_roi_camara(imagen_camara_rgb)
    imagen_roi_gris = cv2.cvtColor(imagen_roi_rgb, cv2.COLOR_RGB2GRAY)
    print(f"ROI seleccionada (x0, y0, x1, y1): {caja_roi}")
    mostrar_imagen(imagen_roi_rgb, "ROI central útil para la actividad")
    mostrar_histograma_gris(imagen_roi_gris, "Histograma de la ROI")
else:
    imagen_roi_rgb = None
    imagen_roi_gris = None
    caja_roi = None
    print("Recorte ROI pendiente: falta la imagen de entrada.")

## Estrategias comparadas

### Estrategia A — Recomendada

**ROI → escala de grises → CLAHE suave → mediana 3×3 → ajuste leve de brillo/contraste**

Parámetros principales:

- `clipLimit = 1.5`
- `tileGridSize = (8, 8)`
- `medianBlur = 3`
- `alpha = 1.2`, `beta = 10`

Se elige como candidata principal porque mejora contraste local sin depender de un aumento global extremo. La mediana posterior reduce ruido resaltado por CLAHE.

### Estrategia B — Alternativa agresiva para descartar/comparar

**ROI → escala de grises → equalizeHist global → Gaussian blur → sharpening suave**

Sirve como comparación porque muestra el riesgo de la ecualización global: puede aumentar contraste, pero también exagerar ruido, banding y halos. No se considera la opción final si destruye naturalidad o inventa bordes.

In [ ]:
def estrategia_a_clahe_suave(imagen_roi_rgb, clip_limit=1.5, tile_grid=(8, 8), alpha=1.2, beta=10):
    gris = cv2.cvtColor(imagen_roi_rgb, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    mejorada = clahe.apply(gris)
    mejorada = cv2.medianBlur(mejorada, 3)
    mejorada = cv2.convertScaleAbs(mejorada, alpha=alpha, beta=beta)
    return cv2.cvtColor(mejorada, cv2.COLOR_GRAY2RGB)


def estrategia_b_ecualizacion_global(imagen_roi_rgb):
    gris = cv2.cvtColor(imagen_roi_rgb, cv2.COLOR_RGB2GRAY)
    ecualizada = cv2.equalizeHist(gris)
    suavizada = cv2.GaussianBlur(ecualizada, (3, 3), 0)
    desenfocada = cv2.GaussianBlur(suavizada, (0, 0), 1.0)
    sharpen_suave = cv2.addWeighted(suavizada, 1.25, desenfocada, -0.25, 0)
    return cv2.cvtColor(sharpen_suave, cv2.COLOR_GRAY2RGB)


if imagen_roi_rgb is not None:
    imagen_estrategia_a = estrategia_a_clahe_suave(imagen_roi_rgb)
    imagen_estrategia_b = estrategia_b_ecualizacion_global(imagen_roi_rgb)

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(imagen_roi_rgb)
    ax[0].set_title("Base comparable: ROI")
    ax[1].imshow(imagen_estrategia_a)
    ax[1].set_title("Estrategia A: CLAHE suave")
    ax[2].imshow(imagen_estrategia_b)
    ax[2].set_title("Estrategia B: ecualización global")
    for eje in ax:
        eje.axis("off")
    plt.tight_layout()
    plt.show()
else:
    imagen_estrategia_a = None
    imagen_estrategia_b = None
    print("Comparación de estrategias pendiente: falta la imagen de entrada.")

## Comparación objetiva y elección final

La comparación no se basa solo en “se ve mejor”. Se reportan métricas sobre la misma ROI para evitar sesgo por tamaño o por fondo eliminado.

**Criterio de selección:**

- La salida final debe aumentar contraste y legibilidad.
- No debe generar halos ni bordes falsos dominantes.
- No debe convertir el ruido/banding en la característica más visible.
- Debe conservar la información importante de la proyección central.

In [ ]:
if imagen_roi_gris is not None:
    gris_a = cv2.cvtColor(imagen_estrategia_a, cv2.COLOR_RGB2GRAY)
    gris_b = cv2.cvtColor(imagen_estrategia_b, cv2.COLOR_RGB2GRAY)

    metricas_roi = calcular_metricas_gris(imagen_roi_gris)
    metricas_a = calcular_metricas_gris(gris_a)
    metricas_b = calcular_metricas_gris(gris_b)

    imprimir_metricas("Base comparable - ROI", metricas_roi)
    imprimir_metricas("Estrategia A - CLAHE suave", metricas_a)
    imprimir_metricas("Estrategia B - ecualización global", metricas_b)

    print("\nLectura técnica:")
    print("- La estrategia A busca mejora moderada de contraste local y control de ruido.")
    print("- La estrategia B puede aumentar más el contraste numérico, pero tiende a exagerar ruido y halos.")
    print("- Por coherencia con el diagnóstico, se selecciona la Estrategia A como resultado final.")
else:
    metricas_roi = None
    metricas_a = None
    metricas_b = None
    print("Comparación objetiva pendiente: falta la imagen de entrada.")

## Resultado final y guardado

Se guarda como salida final la **Estrategia A** porque es la alternativa más coherente con el diagnóstico: corrige contraste local y legibilidad sin intentar recuperar artificialmente el desenfoque óptico.

**Problemas que mejora:**

- subexposición perceptual;
- bajo contraste local;
- legibilidad de la región central;
- ruido fino moderado después de CLAHE.

**Problemas que no resuelve por completo:**

- desenfoque óptico severo;
- información no capturada en la adquisición;
- posible banding vertical estructural;
- límites de señal provocados por poca luz.


In [ ]:
if imagen_estrategia_a is not None:
    imagen_camara_final = imagen_estrategia_a
    guardar_rgb(ruta_final_camara, imagen_camara_final)

    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ax[0].imshow(imagen_roi_rgb)
    ax[0].set_title("Antes: ROI original")
    ax[1].imshow(imagen_camara_final)
    ax[1].set_title("Después: resultado final")
    for eje in ax:
        eje.axis("off")
    plt.tight_layout()
    plt.savefig(ruta_comparacion_camara, dpi=150, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].hist(imagen_roi_gris.ravel(), bins=256, range=(0, 255), color="dimgray")
    ax[0].set_title("Histograma ROI original")
    ax[0].set_xlabel("Intensidad")
    ax[0].set_ylabel("Píxeles")
    ax[1].hist(cv2.cvtColor(imagen_camara_final, cv2.COLOR_RGB2GRAY).ravel(), bins=256, range=(0, 255), color="steelblue")
    ax[1].set_title("Histograma resultado final")
    ax[1].set_xlabel("Intensidad")
    ax[1].set_ylabel("Píxeles")
    for eje in ax:
        eje.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(ruta_histogramas_camara, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Imagen final guardada en: {ruta_final_camara}")
    print(f"Comparación antes/después guardada en: {ruta_comparacion_camara}")
    print(f"Histogramas guardados en: {ruta_histogramas_camara}")
else:
    imagen_camara_final = None
    print("Guardado pendiente: falta la imagen de entrada.")

## Conclusión técnica para defensa oral

La imagen de cámara oscura presenta una degradación dominada por **desenfoque óptico, subexposición y bajo contraste**. El ruido es importante, pero no es el único problema. Por eso el pipeline final no intenta una restauración “milagrosa”: prioriza contraste local y legibilidad mediante un procesamiento suave.

### Elección final

Se elige la **Estrategia A: recorte ROI + grises + CLAHE suave + mediana 3×3 + ajuste leve**.

### Justificación

- El recorte evita que el fondo oscuro domine el análisis.
- CLAHE mejora contraste local en una imagen con iluminación pobre y desigual.
- La mediana 3×3 reduce ruido puntual sin borrar totalmente estructuras.
- El ajuste `alpha=1.2`, `beta=10` es moderado y se aplica después de diagnosticar subexposición.
- La estrategia alternativa con ecualización global se incluye como comparación, pero se descarta como final porque puede exagerar ruido, banding y halos.

### Límites reconocidos

El desenfoque no puede recuperarse completamente en posprocesamiento. Si la captura original no registró detalle fino, ningún filtro puede reconstruirlo de manera objetiva. Con más tiempo, la mejora principal debería hacerse en la adquisición: orificio mejor dimensionado, mayor estabilidad, exposición más larga, ISO bajo, objeto iluminado con luz solar directa y control de filtraciones de luz.

## Plantilla de análisis completada

1. **Descripción de la imagen**
   - Origen: `img_input_tfi1/img-camara-oscura.jpg`.
   - Fotografía propia tomada con cámara oscura, de un objeto real.

2. **Diagnóstico**
   - Problema principal: desenfoque óptico, subexposición y bajo contraste.
   - Problemas secundarios: ruido digital visible y posible banding vertical.
   - Región de interés: zona central donde aparece la proyección.
   - Evidencia: histograma concentrado en niveles bajos/intermedios, bajo rango dinámico y poca separación objeto/fondo.

3. **Objetivo**
   - Mejorar legibilidad visual y contraste local sin introducir pérdida importante de información.

4. **Estrategia A**
   - Pasos: ROI → grises → CLAHE suave → mediana → ajuste leve.
   - Parámetros: `clipLimit=1.5`, `tileGridSize=(8,8)`, `medianBlur=3`, `alpha=1.2`, `beta=10`.
   - Resultado: mejora contraste local y visibilidad central.
   - Ventajas: controlada, reproducible y coherente con bajo contraste/subexposición.
   - Límites: no recupera desenfoque real ni información no capturada.

5. **Estrategia B**
   - Pasos: ROI → grises → ecualización global → gaussiano → sharpening suave.
   - Resultado esperado: mayor contraste aparente, pero más ruido y halos.
   - Ventajas: sirve como comparación y muestra por qué no toda mejora numérica es visualmente correcta.
   - Límites: puede exagerar ruido y banding.

6. **Elección final**
   - Estrategia elegida: Estrategia A.
   - Justificación: mejora legibilidad con menos artefactos y conserva mejor la información importante.
   - Qué resolvió: contraste local y visibilidad de la ROI.
   - Qué no resolvió: desenfoque óptico severo y pérdida de detalle de captura.

7. **Salidas**
   - Imagen final: `img_output_tfi1/img-camara-oscura-final.jpg`.
   - Comparación antes/después: `img_output_tfi1/img-camara-oscura-comparacion.jpg`.
   - Histogramas: `img_output_tfi1/img-camara-oscura-histogramas.jpg`.
